In [1]:
import os
from pathlib import Path
import pandas as pd
import json
from dotenv import load_dotenv
from openai import OpenAI

# move to project root (one level up from notebooks/)
os.chdir(Path.cwd().parent)
print("Working directory:", Path().resolve())

load_dotenv()
client = OpenAI()

raw_dir = Path("data/raw")
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)


Working directory: /Users/maithreyeeiyengar/Library/Mobile Documents/com~apple~CloudDocs/Gen AI Lab/Final Project/GenAI_FinalProject


In [3]:
p1_reviews_df = pd.read_csv(raw_dir / "product1_reviews.csv")
print("Total P1 reviews:", len(p1_reviews_df))
p1_reviews_df


Total P1 reviews: 10


,product_id,review_text
0,P1,I think I'm more than a little red-faced right...
1,P1,"I really liked this product, but I did not lik..."
2,P1,I thoroughly enjoy what Sony has to offer in t...
3,P1,One of the most frustrating device I've EVER u...
4,P1,There's A LOT to love about the Sony WH-1000XM...
5,P1,I previously owned the older version of these ...
6,P1,I recently picked up the Sony WH-1000XM5 headp...
7,P1,"I had the WH-1000XM3 for a long time, until on..."
8,P1,First of all I will start out saying these are...
9,P1,Nicely designed and has a quality feel to it. ...


In [13]:
p2_reviews_df = pd.read_csv(raw_dir / "product2_reviews.csv")
print("Total P2 reviews:", len(p2_reviews_df))
p2_reviews_df

Total P2 reviews: 10


,product_id,review_text
0,P2,Been having this fryer for 2 weeks now and it ...
1,P2,"Wow, this thing is great. So, we finally broke..."
2,P2,I thought the Ninja air fryer would live up to...
3,P2,I love this air fryer! My previous fryer was a...
4,P2,I received this item on 07082025 and was excit...
5,P2,"I've only used this air fryer twice, so still ..."
6,P2,Oh my goodness! I just bought this and have co...
7,P2,Too damn noisy. I had a previous version of th...
8,P2,"The air fryer cooks as described, but I have e..."
9,P2,This my first air fryer. I purchased it on sal...


In [14]:
p3_reviews_df = pd.read_csv(raw_dir / "product3_reviews.csv")
print("Total P3 reviews:", len(p3_reviews_df))
p3_reviews_df

Total P3 reviews: 10


,product_id,review_text
0,P3,I wish I known about Cerave long ago in my tee...
1,P3,I use this product morning and evening and whe...
2,P3,I gave this CeraVe hydrating facial cleanser 5...
3,P3,"Normally I am a fan of cerave products, but no..."
4,P3,This is just me writing of my personal experie...
5,P3,I’ve been using this product for months now. I...
6,P3,I was not trilled with this. I was trying to s...
7,P3,It does not work well. It does not foam. Feels...
8,P3,reviMy initial opinion is that it’s a good pro...
9,P3,"For years, I associated that ""squeaky clean"" f..."


In [15]:
def build_review_prompt(product_name: str, reviews: list[str]):
    joined_reviews = "\n\n---\n\n".join(reviews)

    return [
        {
            "role": "system",
            "content": (
                "You are a careful data analyst helping a product team understand customer reviews. "
                "Extract structured attributes that summarize customer perception, benefits, pain points, "
                "visual cues, and usage contexts."
            )
        },
        {
            "role": "user",
            "content": f"""
Product: {product_name}

Below are customer reviews, separated by '---':

{joined_reviews}

Return ONLY a JSON object with:
- summary (short paragraph)
- key_benefits (list)
- pain_points (list)
- visual_attributes (list)
- usage_contexts (list)
- tone_words (list)

Do NOT output explanation. ONLY return valid JSON.
"""
        }
    ]


In [5]:
product1_name = "Sony WH-1000XM5 Wireless Noise-Canceling Headphones"
p1_reviews = p1_reviews_df["review_text"].tolist()

messages = build_review_prompt(product1_name, p1_reviews)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.3
)

p1_raw = response.choices[0].message.content
print(p1_raw[:500])


```json
{
  "summary": "The Sony WH-1000XM5 headphones are praised for their outstanding noise cancellation, sound quality, and comfort, but they also face significant criticism due to connectivity issues, build quality concerns, and inconsistent performance over time.",
  "key_benefits": [
    "Excellent active noise cancellation (ANC)",
    "High sound quality with clear highs and deep bass",
    "Comfortable and lightweight design",
    "Long battery life with fast charging",
    "Customizabl


In [16]:
product2_name = "Ninja AF101 4-Quart Air Fryer"
p2_reviews = p2_reviews_df["review_text"].tolist()

messages = build_review_prompt(product2_name, p2_reviews)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.3
)

p2_raw = response.choices[0].message.content
print(p2_raw[:500])

```json
{
  "summary": "The Ninja AF101 4-Quart Air Fryer receives mixed reviews from customers, with many praising its cooking speed and ease of use, while others express concerns about its capacity and noise level. Overall, it is appreciated for producing crispy food with less mess compared to traditional frying methods.",
  "key_benefits": [
    "Cooks food quickly",
    "Produces crispy results",
    "Easy to clean",
    "Compact size fits under cabinets",
    "Versatile cooking options (air


In [17]:
product3_name = "CeraVe Hydrating Facial Cleanser, 16 oz"
p3_reviews = p3_reviews_df["review_text"].tolist()

messages = build_review_prompt(product3_name, p3_reviews)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.3
)

p3_raw = response.choices[0].message.content
print(p3_raw[:500])

{
  "summary": "CeraVe Hydrating Facial Cleanser is a gentle, creamy facial cleanser that effectively cleanses without drying the skin. While many users appreciate its hydrating properties and suitability for sensitive skin, some report issues with its effectiveness in removing makeup and providing a 'clean' feeling.",
  "key_benefits": [
    "Gentle and non-drying formula",
    "Hydrates and soothes the skin",
    "Fragrance-free, suitable for sensitive skin",
    "Contains beneficial ingredien


In [19]:
import re
import json

def extract_json(text: str) -> dict:
    """
    Tries to extract a JSON object from a model response.
    Handles cases with ```json fences or extra explanation text.
    """
    text = text.strip()

    # Case 1: fenced code block ```json ... ```
    if text.startswith("```"):
        # Remove the first ```... line
        text = re.sub(r"^```[a-zA-Z0-9]*\s*", "", text)
        # Remove the last ```
        if text.endswith("```"):
            text = text[: -3].strip()

    # Case 2: extra text before or after JSON -> grab from first { to last }
    start = text.find("{")
    end = text.rfind("}") + 1
    if start != -1 and end != -1:
        text = text[start:end]

    # Now parse
    return json.loads(text)


In [20]:
p1_json = extract_json(p1_raw)

out_path = processed_dir / "product1_analysis.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(p1_json, f, indent=2)

out_path


PosixPath('data/processed/product1_analysis.json')

In [29]:
p2_json = extract_json(p2_raw)
out_path = processed_dir / "product2_analysis.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(p2_json, f, indent=2)

out_path



PosixPath('data/processed/product2_analysis.json')

In [30]:
p3_json = extract_json(p3_raw)
out_path = processed_dir / "product3_analysis.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(p3_json, f, indent=2)

out_path

PosixPath('data/processed/product3_analysis.json')

In [33]:
# with open("data/processed/product3_analysis.json", "r", encoding="utf-8") as f:
#     data = json.load(f)

# data.keys(), data["summary"]